In [5]:
import os
import fitz  # PyMuPDF

def force_compress_to_under_1mb(input_path):
    dir_name = os.path.dirname(input_path)
    file_name = os.path.basename(input_path)
    output_path = os.path.join(dir_name, f"final_fixed_{file_name}")

    # Open the heavy PDF
    doc = fitz.open(input_path)
    new_doc = fitz.open()

    print("Re-rendering pages using JPG compression...")

    for page_num in range(len(doc)):
        page = doc.load_page(page_num)
        
        # Render page to a Pixmap (Image)
        # 150 DPI is great for passports; 100-120 if you need it even smaller.
        pix = page.get_pixmap(dpi=150)
        
        # Convert to JPG bytes (which your version supports!)
        # We use quality=70 to ensure it stays under 1MB
        img_bytes = pix.tobytes("jpg", jpg_quality=70)
        
        # Create a temporary PDF page from these image bytes
        img_doc = fitz.open("jpg", img_bytes)
        pdf_bytes = img_doc.convert_to_pdf()
        img_doc.close()
        
        # Add this compressed page to our final document
        temp_page_doc = fitz.open("pdf", pdf_bytes)
        new_doc.insert_pdf(temp_page_doc)
        temp_page_doc.close()

    # Final save with cleanup
    new_doc.save(
        output_path,
        garbage=4,
        deflate=True
    )
    new_doc.close()
    doc.close()

    # Success stats
    orig_size = os.path.getsize(input_path) / (1024 * 1024)
    final_size = os.path.getsize(output_path) / (1024 * 1024)
    
    print(f"\n--- Done! ---")
    print(f"Original: {orig_size:.2f} MB")
    print(f"Compressed: {final_size:.2f} MB")
    print(f"Saved to: {output_path}")

path_to_file = r"C:\Users\bkgha\OneDrive\Desktop\PR\Bala Krishna Ghanta\New Passport.pdf"

if __name__ == "__main__":
    force_compress_to_under_1mb(path_to_file)

Re-rendering pages using JPG compression...

--- Done! ---
Original: 1.25 MB
Compressed: 0.14 MB
Saved to: C:\Users\bkgha\OneDrive\Desktop\PR\Bala Krishna Ghanta\final_fixed_New Passport.pdf
